In [3]:
!pip install youtube-transcript-api

In [4]:
!pip install -U langchain-text-splitters

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()





/var/www/html/myproject/GENAI-Series/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_20550/1541534096.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [6]:
!pip install -U langchain-groq

In [7]:
from transformers import pipeline
from langchain_groq import ChatGroq
import torch

In [8]:
!pip install accelerate

In [9]:
 !pip show youtube-transcript-api

Name: youtube-transcript-api
Version: 1.2.4
Summary: This is a python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: 
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: MIT
Location: /var/www/html/myproject/GENAI-Series/venv/lib/python3.10/site-packages
Requires: defusedxml, requests
Required-by: 


In [10]:
!pip install langchain-text-splitters langchain-huggingface langchain-community faiss-cpu langchain-groq sentence-transformers

In [11]:

## Document Loader

yt_api = YouTubeTranscriptApi()

video_id = "SfOaZIGJ_gs"
transcript = yt_api.fetch(video_id=video_id)

transcript_text = " ".join([doc.text for doc in transcript])

## Text Splitting

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap= 50)
chunks = splitter.split_text(transcript_text)

## Call embedding models

embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

## Vector Store

vector_store = FAISS.from_texts(chunks,embeddings)

vector_store.similarity_search('Tell me about Ai')


## Retriever

retriever = vector_store.as_retriever(search_kwargs={"k":4})
retriever.invoke("How to start a company")


## Augmentation 
prompt = PromptTemplate(template=""" You are a helpful Assitant Annwer from the following context. If the context is insufficient then directly say I Dont know {context} Question {question} """, input_variables=['context','question'])

question = "How to start a compnay has been spoken in this video ? "

context = retriever.invoke(question)

context = " ".join(i.page_content for i in context)
final_prompt = prompt.invoke({'question':question, 'context':context})


## Generation

# model_id = "openai/gpt-oss-20b"
model_id = "openai/gpt-oss-120b"

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

response = llm.invoke(final_prompt)
print(response.content)







IpBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=SfOaZIGJ_gs! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide th